In [14]:
#!pip install bitsandbytes
#!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.0.7/flash_attn-2.0.7+cu121torch2.1cxx11abiFALSE-cp310-cp310-linux_x86_64.whl
#!pip uninstall flash_attn
#!FLASH_ATTENTION_FORCE_BUILD=TRUE pip install flash-attn
#!pip install optimum
#!pip install autoawq
#!pip install vllm==0.3.3

In [15]:
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig, LlavaForConditionalGeneration
import torch
from PIL import Image
import requests

In [16]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    #bnb_4bit_compute_dtype=bfloat16  # Computation type, from torch import bfloat16
)

In [17]:
#processor = LlavaNextProcessor.from_pretrained("TheBloke/llava-v1.5-13B-AWQ")
#LlavaNextForConditionalGeneration  LlavaForConditionalGeneration
# circulus/llava-v1.6-mistral-7b-awq
#model = LlavaNextForConditionalGeneration.from_pretrained("liuhaotian/llava-v1.5-7b", low_cpu_mem_usage=True)# , quantization_config=quantization_config) 

In [18]:
#model.to("cuda")

In [19]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader
import os
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer
import sglang as sgl
import json
from datetime import datetime

In [20]:
# load as image documents
image_documents = SimpleDirectoryReader("/home/data/ISIC2019/seg", required_exts=[".jpeg", ".jpg", ".png"], recursive=True, filename_as_id=True).load_data()

In [21]:
def metadata_str2dict(string):
    # Split the string into lines
    lines = string.strip().split("\n")
    
    # Create a dictionary to store the key-value pairs
    file_info_dict = {}
    
    # Parse each line and add key-value pairs to the dictionary
    for line in lines:
        key, value = line.split(": ", 1)
        file_info_dict[key] = value
    
    return(file_info_dict)

In [22]:
image_patch = []
image_no_patch = []

for image in image_documents:
    metadata = metadata_str2dict(image.get_metadata_str())
    image_path = metadata['file_path']

    folder_name = os.path.basename(os.path.dirname(image_path))
    
    if folder_name == "patch":
        metadata['patch'] = True
        image_patch.append({'image': image, 'meta': metadata})


    elif folder_name == "no_patch":
        metadata['patch'] = False
        image_no_patch.append({'image': image, 'meta': metadata})


    else:
        print("The jpg is not in either the patch or no_patch folder.")

In [23]:
print(image_patch[0]['meta'])

{'file_path': '/home/data/ISIC2019/seg/patch/ISIC_0000352.jpg', 'file_name': 'ISIC_0000352.jpg', 'file_type': 'image/jpeg', 'file_size': '14687', 'creation_date': '2024-05-20', 'last_modified_date': '2024-05-20', 'patch': True}


In [24]:
@sgl.function
def image_qa(s, image_file, question):
    s += sgl.user(sgl.image(image_file) + question)
    s += sgl.assistant(sgl.gen("answer", max_tokens=30))

@sgl.function
def image_cot(s, image_file, question1, question2):
    s += sgl.user(sgl.image(image_file) + question1)
    s += sgl.assistant(sgl.gen("answer1", max_tokens=256))
    s += sgl.user(question2)
    s += sgl.assistant(sgl.gen("answer2", max_tokens=30))


os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

#python3 -m sglang.launch_server --model-path liuhaotian/llava-v1.6-vicuna-7b --tokenizer-path llava-hf/llava-1.5-7b-hf --chat-template vicuna_v1.1 --port 30000

model_path = "liuhaotian/llava-v1.6-vicuna-7b" # can I get it from backend?
tokenizer_path = "llava-hf/llava-1.5-7b-hf"
chat_template = "chat-template vicuna_v1.1"

sgl.set_default_backend(sgl.RuntimeEndpoint("http://localhost:30000")) #30000


In [25]:
num_imgs = 100
num_imgs_per_class = int(np.ceil(num_imgs/2))

# combine images with and without patch
images = []
images.extend(image_patch[:num_imgs_per_class])
images.extend(image_no_patch[:num_imgs_per_class])

# output file
data_dict = {}
data_dict['model_path'] = model_path
data_dict['tokenizer_path'] = tokenizer_path
data_dict['chat_template'] = chat_template
data_dict['num_imgs'] = num_imgs


In [26]:
def run_inference(images, num_imgs, question1, question2=None, data_dict=None):
    correct_classified = 0
    false_classified = 0
    more_than_one_word = 0
    num_yes_patch = 0
    num_no_no_patch = 0
    num_yes = 0
    num_no = 0

    output_dict = None
    if data_dict is not None:
        if question2 is None:
            data_dict['prompt']= question1
        else:
            data_dict['question1']= question1
            data_dict['question2']= question2
        output_dict = {}


    print("-----------------------------------------\n")
    
    print(question1)
    if question2 is not None:
        print('\n', question2)
    
    for i, image_dict in enumerate(tqdm(images[:num_imgs])):
        image = image_dict['image']
        meta = image_dict['meta']
        
        ## display image
        #imageUrl = meta['file_path']
        #image = Image.open(imageUrl).convert("RGB")
        #plt.figure(figsize=(8, 3))
        #plt.imshow(image)
    
        # image
        image_path = meta['file_path']

        #response
        if question2 is None:
            state = image_qa.run(
            image_file=image_path,
            question=question1,
            max_new_tokens=128, 
            temperature=0)
            #print(state["answer"], "\n")
            response = state["answer"].lower().replace(" ", "")
        else:
            state = image_cot.run(
            image_file=image_path,
            question1=question1,
            question2=question2,
            max_new_tokens=128,
            temperature=0)
            print("A1: ", state["answer1"], "\n")
            print("A2: ",state["answer2"], "\n")
            response = state["answer2"].lower().replace(" ", "")
        
        
        if response in {"yes", "yes."}:
            #print("yes")
            num_yes += 1
            if meta['patch']:
                correct_classified += 1
                num_yes_patch += 1
                #print("correct")
            else:
                false_classified += 1
                #print("false")
    
        elif response in {"no", "no."}:
            #print("no")
            
            num_no += 1
            if meta['patch']:
                false_classified += 1
                #print("false")
            else:
                correct_classified += 1
                num_no_no_patch +=1
                #print("correct")
        else:
            #print("No yes or no!")
            more_than_one_word += 1

        # save data
        if output_dict is not None:
            img_output_dict = {}
            img_output_dict['img_meta'] = image_dict['meta']
            if question2 is None:
                img_output_dict['answer'] = state["answer"] 
            else:
                img_output_dict['answer1'] = state["answer1"] 
                img_output_dict['answer2'] = state["answer2"]
            output_dict[f'img_{i}'] = img_output_dict


    acc = correct_classified/(num_imgs - more_than_one_word)
    
    print("\nAccuracy: ",acc)
    print("correct classified: ", correct_classified,"\nfalse classified: ", false_classified, 
          "\nNumber yes: ", num_yes, "\nNumber no: ", num_no, "\nNeither yes or no: ", more_than_one_word,
          "\nNumber True positives: ", num_yes_patch, "\nNumber True negatives: ", num_no_no_patch,
          "\nnum images: ", num_imgs)
    print("\n-----------------------------------------\n\n")

    # save data
    if data_dict is not None:
        data_dict['accuracy'] = acc
        data_dict['correct_classified'] = correct_classified
        data_dict['false_classified'] = false_classified
        data_dict['num_yes'] = num_yes
        data_dict['num_no'] = num_no
        data_dict['more_than_one_word'] = more_than_one_word
        data_dict['true_positive'] = num_yes_patch
        data_dict['true_negative'] = num_no_no_patch
        data_dict['outputs']=output_dict

        # Mkdir if not existent
        path="/home/outputfiles"
        os.makedirs(path, exist_ok=True)
        
        # Generate a timestamp
        timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")

        # Create a filename with the timestamp
        filename = f"{path}/llava_output_{str(correct_classified)}_{timestamp}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data_dict, f, indent=4)
        

In [27]:
question1 = "The image shows a segmented close-up from a skin lesion. As it is a segmetation map the image does not contain the colors of the original image. Is there something on the image that has a smooth elliptic or circular shape that could be a patch attached next to a lesion? Please explain."
question2 = "Can you tell me, considering your explanation, if you see a smooth elliptic or circular shaped patch attached on the skin or not? Answer with one word of the options [yes, no]."
run_inference(images, num_imgs=100, question1=question1, question2=question2, data_dict=data_dict)



-----------------------------------------

The image shows a segmented close-up from a skin lesion. As it is a segmetation map the image does not contain the colors of the original image. Is there something on the image that has a smooth elliptic or circular shape that could be a patch attached next to a lesion? Please explain.

 Can you tell me, considering your explanation, if you see a smooth elliptic or circular shaped patch attached on the skin or not? Answer with one word of the options [yes, no].


  1%|▊                                                                                    | 1/100 [00:05<09:51,  5.98s/it]

A1:   The image you've provided appears to be a segmentation map, which is typically used in medical imaging to identify and differentiate between various tissue types within a lesion. The segmentation map is not a color image, so it does not contain the original colors of the lesion.

In the segmentation map, there is a smooth elliptical or circular shape that could be interpreted as a patch or a lesion that has been segmented out from the surrounding tissue. This shape is distinct from the surrounding tissue, which is segmented into different areas, possibly indicating different types of tissue or different stages of a lesion.

The segmentation map is a tool used by medical professionals to analyze and diagnose skin lesions. It helps to identify the extent of the lesion, the type of tissue affected, and the progression of the lesion. The patch-like shape could be a focus of interest for further analysis or treatment planning. 

A2:   Yes. 



  2%|█▋                                                                                   | 2/100 [00:12<09:49,  6.01s/it]

A1:   The image you've provided appears to be a segmentation map, which is typically used in medical imaging to identify and differentiate between various tissue types within a lesion. The segmentation map is not a color image, so it does not contain the colors of the original image.

In the segmentation map, there are various shapes that could be interpreted as different tissue types or segments within the lesion. However, without additional context or information about the original image, it's difficult to determine the exact nature of these shapes.

Regarding the smooth elliptical or circular shape that could be a patch attached next to the lesion, this is a possibility, but it's not clear from the segmentation map alone. The segmentation map does not provide enough detail to confidently identify the nature of the shape or its relationship to the lesion.

If you have access to the original image or more detailed information about the segmentation map, it might be possible to provide

...

A1:   The image you've provided appears to be a segmentation map, which is typically used in medical imaging to identify and differentiate between various tissue types within a lesion. The segmentation map is not a color image, so it does not contain the colors of the original image.

In the segmentation map, there are various shapes that could be interpreted as different tissue types or segments within the lesion. However, without additional context or information about the original image, it's difficult to determine the exact nature of these shapes.

Regarding the smooth elliptical or circular shape, it could be a patch that has been attached to the lesion. This might be a part of a treatment or a diagnostic procedure. The patch could be a biological dressing, a medical device, or a part of a surgical procedure.

Without more information, it's not possible to provide a definitive answer about the nature of the patch. If you have more context or additional images, I might be able to p

 99%|███████████████████████████████████████████████████████████████████████████████████▏| 99/100 [10:09<00:06,  6.33s/it]

A1:   The image you've provided appears to be a segmentation map, which is a visual representation of a medical image, such as a skin lesion, where different regions of the image are color-coded to represent different tissue types or conditions. In this case, the colors are not the actual colors of the original image, but rather a way to differentiate between different segments of the lesion.

Regarding the smooth elliptical or circular shape that could be a patch attached next to the lesion, it's not immediately clear from the segmentation map alone. The segmentation map does not provide enough detail to determine the exact nature of the shape or its relation to the lesion.

However, in medical imaging, it is common to see various shapes and patterns that could represent different tissue types, such as healthy skin, inflammation, or lesions. If the shape is indeed a patch attached next to the lesion, it would typically be a part of the medical treatment or assessment of the lesion.

T

100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [10:15<00:00,  6.15s/it]

A1:   The image you've provided appears to be a segmentation map, which is typically used in medical imaging to identify and differentiate between various tissue types within a lesion. The map is segmented into different colors, which are likely to represent different tissue types or characteristics.

In the image, there is a smooth elliptical or circular shape that could be interpreted as a patch or a lesion that has been segmented out from the surrounding tissue. This patch might be a part of the original image that has been isolated for further analysis or to be compared with other segments.

The segmentation map is a common tool in medical imaging to aid in the diagnosis and treatment of skin lesions. It allows for a detailed analysis of the tissue types and can be used to identify areas of interest for further examination or treatment. The patch you're referring to could be a part of this analysis, where the segmentation map has been used to isolate a specific area for closer exam

In [28]:
for prompt in prompts:
    run_inference(images, num_imgs=100, question1=prompt, question2=None, data_dict=data_dict)
    

NameError: name 'prompts' is not defined